# Tests Machine Learning Model

## 📚 Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import ast
import re
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

## 💾 Import Data

In [ ]:
data = pd.read_csv("../raw_data/recipes_data.csv")

## 📐 Métriques d'évaluation

Le modèle `NearestNeighbors` est un modèle de **recommandation/retrieval**, pas de classification.  
Utiliser l'exact match sur le titre est biaisé car :
- Deux recettes différentes peuvent partager les mêmes ingrédients
- Le modèle n'a pas vocation à retrouver la *même* recette, mais des recettes *similaires*

**Métriques utilisées :**
- **Cosine similarity moyenne @top-1** : proximité dans l'espace TF-IDF (1 = identique, 0 = aucun ingrédient commun)
- **Jaccard Hit Rate @5** : pour chaque recette test, est-ce qu'au moins un des 5 résultats partage ≥20% des ingrédients ?

In [ ]:
def parse_ner(row):
    try:
        return ast.literal_eval(row)
    except (ValueError, SyntaxError):
        return []

def jaccard(a, b):
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)

def evaluate(distances, indices, train_ingredient_sets, test_ingredient_sets, threshold=0.2):
    """Calcule les deux métriques pour un modèle NearestNeighbors."""
    # Métrique 1 : cosine similarity moyenne du top-1
    mean_sim = 1 - distances[:, 0].mean()

    # Métrique 2 : Jaccard Hit Rate @5
    hits = sum(
        1 for i, neighbors in enumerate(indices)
        if any(
            jaccard(test_ingredient_sets[i], train_ingredient_sets[j]) >= threshold
            for j in neighbors
        )
    )
    hit_rate = hits / len(indices)

    print(f"  Cosine similarity moyenne @top-1 : {mean_sim:.3f}")
    print(f"  Jaccard Hit Rate @5 (seuil={threshold}) : {hit_rate:.1%}")
    return mean_sim, hit_rate

## 1️⃣ Modèle 1 — NER brut (non nettoyé)

### ✂ Split Dataset

In [ ]:
min_data = data.iloc[:20000].copy()

train_df1, test_df1 = train_test_split(min_data, test_size=0.3, random_state=42)

X1_train = train_df1["NER"]
X1_test  = test_df1["NER"]

print(f"Train : {len(X1_train)} recettes | Test : {len(X1_test)} recettes")

### ↗ Vectorisation

In [ ]:
vectorizer1 = TfidfVectorizer()
X1_train_vec = vectorizer1.fit_transform(X1_train)
X1_test_vec  = vectorizer1.transform(X1_test)

### 🤖 Modèle

In [ ]:
model1 = NearestNeighbors(n_neighbors=5, metric="cosine")
model1.fit(X1_train_vec)

### 🧪 Prédictions

In [ ]:
distances1, indices1 = model1.kneighbors(X1_test_vec)

### 💯 Score

In [ ]:
# Construire les ensembles d'ingrédients pour le calcul Jaccard
train1_sets = [set(parse_ner(r)) for r in X1_train.values]
test1_sets  = [set(parse_ner(r)) for r in X1_test.values]

print("Modèle 1 (NER brut) :")
evaluate(distances1, indices1, train1_sets, test1_sets)

## 2️⃣ Modèle 2 — NER nettoyé

### 🧽 Nettoyage des ingrédients

#### 📋 Variables globales

In [ ]:
KNOWN_BRANDS = {
    "cool whip", "velveeta", "crisco", "jello", "jell-o", "kraft",
    "bisquick", "campbells", "campbell", "heinz", "hellmanns", "hellmann",
    "karo", "lipton", "miracle whip", "pillsbury", "realemon", "tabasco",
    "worcestershire", "spam", "oreo", "ritz", "swanson", "rotel", "ro-tel",
    "duncan hines", "betty crocker", "knorr", "maggi", "frenchs",
    "hidden valley", "pace", "bushs", "chef boyardee", "progresso",
    "del monte", "green giant", "quaker", "philadelphia",
    "dorito", "doritos", "vegenaise", "squirt", "campbells golden",
}

NON_FOOD = {
    "grill pan", "cookie cutters", "cookie cutter", "cooking spray",
    "wet ingredients", "dry ingredients", "kind deeds", "medley",
    "pan spray", "nonstick spray",
}

MIN_FREQUENCY = 200

#### 🐍 Fonctions

In [ ]:
def clean_ingredient(ingredient):
    if not ingredient or pd.isna(ingredient):
        return None
    cleaned = ingredient.strip().lower()
    cleaned = re.sub(r"[^a-z\s\-]", "", cleaned)
    cleaned = re.sub(r"^(a|an|the|some)\s+", "", cleaned).strip()
    if len(cleaned) <= 2:
        return None
    return cleaned

def is_valid(ing):
    if ing in KNOWN_BRANDS or ing in NON_FOOD:
        return False
    if re.search(r'\b(de|of|with|and|or|a|the|in|for)$', ing):
        return False
    if ing.startswith("-") or ing.endswith("-"):
        return False
    return True

#### 🔢 Calcul du vocabulaire valide

> Le comptage est fait **une seule fois** sur `min_data` puis réutilisé pour chaque recette.

In [ ]:
# Calcul du compteur une seule fois (et non à chaque appel de la fonction)
ingredient_counter = Counter()
for ner_row in min_data["NER"]:
    for ing in parse_ner(ner_row):
        cleaned = clean_ingredient(ing)
        if cleaned:
            ingredient_counter[cleaned] += 1

valid_ingredients = {
    ing for ing, n in ingredient_counter.items()
    if n >= MIN_FREQUENCY and is_valid(ing)
}

print(f"{len(valid_ingredients)} ingrédients valides trouvés")

In [ ]:
# valid_ingredients est capturé par closure — pas recalculé à chaque ligne
def get_clean_ingredients(ner_row):
    return [
        ing for raw in parse_ner(ner_row)
        if (ing := clean_ingredient(raw)) and ing in valid_ingredients
    ]

min_data = min_data.copy()
min_data["clean_ingredients"] = min_data["NER"].apply(get_clean_ingredients)
min_data["clean_ingredients_str"] = min_data["clean_ingredients"].apply(lambda x: " ".join(x))

# Vérification rapide
print(min_data[["NER", "clean_ingredients"]].head(3).to_string())

### ✂ Split Dataset

In [ ]:
train_df2, test_df2 = train_test_split(min_data, test_size=0.3, random_state=42)

X2_train = train_df2["clean_ingredients_str"]
X2_test  = test_df2["clean_ingredients_str"]

print(f"Train : {len(X2_train)} recettes | Test : {len(X2_test)} recettes")

### ↗ Vectorisation

In [ ]:
vectorizer2 = TfidfVectorizer()
X2_train_vec = vectorizer2.fit_transform(X2_train)
X2_test_vec  = vectorizer2.transform(X2_test)

### 🤖 Modèle

In [ ]:
model2 = NearestNeighbors(n_neighbors=5, metric="cosine")
model2.fit(X2_train_vec)

### 🧪 Prédictions

In [ ]:
distances2, indices2 = model2.kneighbors(X2_test_vec)

### 💯 Score

In [ ]:
train2_sets = [set(x) for x in train_df2["clean_ingredients"].values]
test2_sets  = [set(x) for x in test_df2["clean_ingredients"].values]

print("Modèle 2 (NER nettoyé) :")
evaluate(distances2, indices2, train2_sets, test2_sets)

## 📊 Comparaison des deux modèles

In [ ]:
sim1, hr1 = evaluate(distances1, indices1, train1_sets, test1_sets)
sim2, hr2 = evaluate(distances2, indices2, train2_sets, test2_sets)

comparison = pd.DataFrame(
    {
        "Modèle 1 — NER brut": [round(sim1, 3), f"{hr1:.1%}"],
        "Modèle 2 — NER nettoyé": [round(sim2, 3), f"{hr2:.1%}"],
    },
    index=["Cosine similarity @top-1", "Jaccard Hit Rate @5 (seuil=0.2)"],
)
comparison

**Interprétation** :
- **Cosine similarity plus élevée** → le modèle trouve des voisins plus proches dans l'espace des ingrédients
- **Jaccard Hit Rate plus élevé** → plus souvent au moins un résultat pertinent dans le top-5